# 01 — Data Collection

## The drivers of economic growth in Europe: evidence from productivity and investment

**Research question:** How are productivity growth and investment associated with economic growth across European countries?

This notebook imports four public datasets, selects European economies, reshapes the data into country–year format, and creates one collected panel. The output remains deliberately broad; the next notebook selects the final analysis period and handles missing observations.

### Notebook output

`data/processed/europe_panel_collected.csv`

In [ ]:
# Core libraries
from pathlib import Path

import pandas as pd

In [ ]:
# Locate the project folder whether the notebook is launched from the
# project root or from the notebooks folder.
project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

raw_dir = project_dir / "data" / "raw"
processed_dir = project_dir / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)


def find_raw_file(*candidate_names):
    """Return the first matching raw-data file or show a helpful error."""
    for name in candidate_names:
        candidate = raw_dir / name
        if candidate.exists():
            return candidate

    expected = ", ".join(candidate_names)
    raise FileNotFoundError(
        f"No matching file was found in {raw_dir}. Expected one of: {expected}"
    )


# Both the short project names and the original download names are accepted.
gdp_file = find_raw_file(
    "gdp_growth.csv",
    "API_NY.GDP.MKTP.KD.ZG_DS2_en_csv_v2_57.csv"
)
investment_file = find_raw_file(
    "investment.csv",
    "API_NE.GDI.TOTL.ZS_DS2_en_csv_v2_140.csv"
)
education_file = find_raw_file(
    "education.csv",
    "API_SE.XPD.TOTL.GD.ZS_DS2_en_csv_v2_366.csv"
)
productivity_path = find_raw_file("productivity.xlsx", "WB-ASPD.xlsx")

print("Project folder:", project_dir)
print("Raw files found:")
for file_path in [gdp_file, investment_file, education_file, productivity_path]:
    print("-", file_path.name)

## 1. Data sources and analytical roles

| Dataset | Variable used | Unit | Role |
|---|---|---:|---|
| World Bank WDI — `NY.GDP.MKTP.KD.ZG` | GDP growth | Annual % | Dependent variable |
| World Bank WDI — `NE.GDI.TOTL.ZS` | Gross capital formation | % of GDP | Main explanatory variable |
| World Bank ASPD — `WB.ASPD.dlpe` | Labour productivity growth | Annual % | Main explanatory variable |
| World Bank WDI — `SE.XPD.TOTL.GD.ZS` | Government education expenditure | % of GDP | Supplementary variable |

Education is collected for descriptive context, but it is not part of the balanced main model because its country–year coverage is incomplete.

## 2. GDP growth

The World Bank file contains countries, territories, and regional aggregates in wide format. We verify the indicator, retain the European country codes defined below, and reshape the year columns into long country–year format.

In [109]:
gdp_growth = pd.read_csv(gdp_file, skiprows=4)

print("Dataset dimensions:", gdp_growth.shape)
display(gdp_growth.head())

Dataset dimensions: (265, 71)


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,3.493430,3.212471,1.229144,-23.940059,14.833692,10.607961,9.516360,6.810777,NaN,NaN
1,Africa Eastern and Southern,AFE,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,NaN,0.418605,7.937089,5.624317,4.650349,5.138153,...,2.678863,2.706207,1.935593,-2.930764,4.452643,3.668029,1.941382,2.787931,3.743016,NaN
2,Afghanistan,AFG,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,2.647003,1.189228,3.911603,-2.351101,-20.738839,-6.240172,2.266944,1.873193,NaN,NaN
3,Africa Western and Central,AFW,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,NaN,1.869593,3.726016,7.038172,5.364580,4.105660,...,2.302761,2.899954,3.283142,-3.718793,2.545473,4.477108,3.644591,4.596183,4.600100,NaN
4,Angola,AGO,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.164689,-0.506698,-1.055234,-5.273491,1.090584,3.557492,1.320166,4.952679,3.133535,NaN


### Initial validation

The indicator code and calendar columns are checked before any filtering or reshaping.

In [110]:
# Basic structure
print("Number of rows:", gdp_growth.shape[0])
print("Number of columns:", gdp_growth.shape[1])

# Identify the year columns
year_columns = [
    column for column in gdp_growth.columns
    if str(column).isdigit()
]

print("\nFirst year:", year_columns[0])
print("Last year:", year_columns[-1])
print("Number of years:", len(year_columns))

# Verify the indicator
print("\nIndicator:")
print(gdp_growth["Indicator Name"].unique())

print("\nIndicator code:")
print(gdp_growth["Indicator Code"].unique())

Number of rows: 265
Number of columns: 71

First year: 1960
Last year: 2025
Number of years: 66

Indicator:
<StringArray>
['GDP growth (annual %)']
Length: 1, dtype: str

Indicator code:
<StringArray>
['NY.GDP.MKTP.KD.ZG']
Length: 1, dtype: str


### European country definition

The sample uses individual European economies rather than World Bank regional aggregates. A shared list of ISO3 codes ensures that the same geographical filter is applied to every dataset. The collected panel contains 46 economies; five are later excluded from the balanced core sample because productivity data are unavailable.

In [111]:
# World Bank country codes for the European sample
europe_codes = [
    "ALB", "AND", "AUT", "BEL", "BGR", "BIH", "BLR",
    "CHE", "CYP", "CZE", "DEU", "DNK", "ESP", "EST",
    "FIN", "FRA", "GBR", "GRC", "HRV", "HUN", "IRL",
    "ISL", "ITA", "LIE", "LTU", "LUX", "LVA", "MCO",
    "MDA", "MKD", "MLT", "MNE", "NLD", "NOR", "POL",
    "PRT", "ROU", "RUS", "SMR", "SRB", "SVK", "SVN",
    "SWE", "TUR", "UKR", "XKX"
]

# Filter the GDP dataset
gdp_europe = gdp_growth[
    gdp_growth["Country Code"].isin(europe_codes)
].copy()

print("Number of European countries:", gdp_europe.shape[0])

display(
    gdp_europe[["Country Name", "Country Code"]]
    .sort_values("Country Name")
    .reset_index(drop=True)
)

Number of European countries: 46


,Country Name,Country Code
0,Albania,ALB
1,Andorra,AND
2,Austria,AUT
3,Belarus,BLR
4,Belgium,BEL
5,Bosnia and Herzegovina,BIH
6,Bulgaria,BGR
7,Croatia,HRV
8,Cyprus,CYP
9,Czechia,CZE


### Reshaping the GDP growth data

We convert the dataset from wide format to panel format, with one observation for each country and year.

In [112]:
# Convert the year columns from wide to long format
gdp_europe_long = gdp_europe.melt(
    id_vars=["Country Name", "Country Code"],
    value_vars=year_columns,
    var_name="Year",
    value_name="GDP Growth"
)

# Convert year from text to integer
gdp_europe_long["Year"] = gdp_europe_long["Year"].astype(int)

# Sort the observations
gdp_europe_long = (
    gdp_europe_long
    .sort_values(["Country Name", "Year"])
    .reset_index(drop=True)
)

print("Dataset dimensions:", gdp_europe_long.shape)
display(gdp_europe_long.head(10))

Dataset dimensions: (3036, 4)


,Country Name,Country Code,Year,GDP Growth
0,Albania,ALB,1960,NaN
1,Albania,ALB,1961,NaN
2,Albania,ALB,1962,NaN
3,Albania,ALB,1963,NaN
4,Albania,ALB,1964,NaN
5,Albania,ALB,1965,NaN
6,Albania,ALB,1966,NaN
7,Albania,ALB,1967,NaN
8,Albania,ALB,1968,NaN
9,Albania,ALB,1969,NaN


## 3. Investment

Investment is measured by **gross capital formation as a percentage of GDP**. This includes additions to fixed assets and changes in inventories.

In [113]:
investment = pd.read_csv(investment_file, skiprows=4)

print("Dataset dimensions:", investment.shape)
print("\nIndicator:", investment["Indicator Name"].unique())
print("Indicator code:", investment["Indicator Code"].unique())
display(investment.head())

Dataset dimensions: (265, 71)

Indicator:
<StringArray>
['Gross capital formation (% of GDP)']
Length: 1, dtype: str

Indicator code:
<StringArray>
['NE.GDI.TOTL.ZS']
Length: 1, dtype: str


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,Gross capital formation (% of GDP),NE.GDI.TOTL.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,20.551171,22.401718,22.611388,23.931090,20.258085,19.643757,18.380863,25.745706,NaN,NaN
1,Africa Eastern and Southern,AFE,Gross capital formation (% of GDP),NE.GDI.TOTL.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,22.000905,21.780866,21.771203,19.511949,19.643056,20.460209,19.843989,18.509686,18.192575,NaN
2,Afghanistan,AFG,Gross capital formation (% of GDP),NE.GDI.TOTL.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,11.500000,13.000000,16.700000,15.292482,17.086890,NaN,NaN
3,Africa Western and Central,AFW,Gross capital formation (% of GDP),NE.GDI.TOTL.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Angola,AGO,Gross capital formation (% of GDP),NE.GDI.TOTL.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,28.756030,23.934661,24.536161,19.758245,16.245641,14.506248,12.438866,10.637746,NaN,NaN


### European investment data

We retain the same European economies used for GDP growth and reshape the investment data into country-year panel format.

In [114]:
# Select the European economies
investment_europe = investment[
    investment["Country Code"].isin(europe_codes)
].copy()

# Identify the year columns in this dataset
investment_year_columns = [
    column for column in investment_europe.columns
    if str(column).isdigit()
]

# Convert from wide to long format
investment_europe_long = investment_europe.melt(
    id_vars=["Country Name", "Country Code"],
    value_vars=investment_year_columns,
    var_name="Year",
    value_name="Investment"
)

# Convert year to integer and sort the observations
investment_europe_long["Year"] = investment_europe_long["Year"].astype(int)

investment_europe_long = (
    investment_europe_long
    .sort_values(["Country Name", "Year"])
    .reset_index(drop=True)
)

print("Number of European countries:", investment_europe.shape[0])
print("Period:", investment_year_columns[0], "-", investment_year_columns[-1])
print("Dataset dimensions:", investment_europe_long.shape)

display(investment_europe_long.head(10))

Number of European countries: 46
Period: 1960 - 2025
Dataset dimensions: (3036, 4)


,Country Name,Country Code,Year,Investment
0,Albania,ALB,1960,NaN
1,Albania,ALB,1961,NaN
2,Albania,ALB,1962,NaN
3,Albania,ALB,1963,NaN
4,Albania,ALB,1964,NaN
5,Albania,ALB,1965,NaN
6,Albania,ALB,1966,NaN
7,Albania,ALB,1967,NaN
8,Albania,ALB,1968,NaN
9,Albania,ALB,1969,NaN


## 4. Education expenditure

Government education expenditure is measured as a percentage of GDP. It is retained as a **supplementary variable** because missing observations prevent it from forming part of the balanced main panel.

In [115]:
education = pd.read_csv(education_file, skiprows=4)

print("Dataset dimensions:", education.shape)
print("\nIndicator:", education["Indicator Name"].unique())
print("Indicator code:", education["Indicator Code"].unique())
display(education.head())

Dataset dimensions: (265, 71)

Indicator:
<StringArray>
['Government expenditure on education, total (% of GDP)']
Length: 1, dtype: str

Indicator code:
<StringArray>
['SE.XPD.TOTL.GD.ZS']
Length: 1, dtype: str


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,"Government expenditure on education, total (% ...",SE.XPD.TOTL.GD.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,4.455820,4.548764,4.435037,NaN,3.618558,NaN,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,"Government expenditure on education, total (% ...",SE.XPD.TOTL.GD.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,4.596270,4.909190,4.511410,4.090565,4.368379,3.697668,3.962293,NaN,NaN,NaN
2,Afghanistan,AFG,"Government expenditure on education, total (% ...",SE.XPD.TOTL.GD.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,4.343190,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Africa Western and Central,AFW,"Government expenditure on education, total (% ...",SE.XPD.TOTL.GD.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,3.120130,3.006575,3.000010,3.336760,3.096926,2.891687,3.215620,NaN,NaN,NaN
4,Angola,AGO,"Government expenditure on education, total (% ...",SE.XPD.TOTL.GD.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,2.466879,2.183513,2.073064,2.667447,2.297197,2.385359,2.512737,NaN,NaN,NaN


### European education expenditure data

We retain the same European economies and reshape education expenditure into country-year panel format.

In [116]:
# Select the European economies
education_europe = education[
    education["Country Code"].isin(europe_codes)
].copy()

# Identify the year columns
education_year_columns = [
    column for column in education_europe.columns
    if str(column).isdigit()
]

# Convert from wide to long format
education_europe_long = education_europe.melt(
    id_vars=["Country Name", "Country Code"],
    value_vars=education_year_columns,
    var_name="Year",
    value_name="Education Expenditure"
)

# Convert year to integer and sort
education_europe_long["Year"] = (
    education_europe_long["Year"].astype(int)
)

education_europe_long = (
    education_europe_long
    .sort_values(["Country Name", "Year"])
    .reset_index(drop=True)
)

print("Number of European countries:", education_europe.shape[0])
print(
    "Period:",
    education_year_columns[0],
    "-",
    education_year_columns[-1]
)
print("Dataset dimensions:", education_europe_long.shape)

display(education_europe_long.head(10))

Number of European countries: 46
Period: 1960 - 2025
Dataset dimensions: (3036, 4)


,Country Name,Country Code,Year,Education Expenditure
0,Albania,ALB,1960,NaN
1,Albania,ALB,1961,NaN
2,Albania,ALB,1962,NaN
3,Albania,ALB,1963,NaN
4,Albania,ALB,1964,NaN
5,Albania,ALB,1965,NaN
6,Albania,ALB,1966,NaN
7,Albania,ALB,1967,NaN
8,Albania,ALB,1968,NaN
9,Albania,ALB,1969,NaN


## 5. Productivity

The World Bank ASPD workbook contains several productivity and production indicators. The project uses `WB.ASPD.dlpe`, the annual labour productivity growth rate.

In [117]:
# Open the productivity workbook and inspect its sheet names.
productivity_workbook = pd.ExcelFile(productivity_path)

print("Available sheets:")
print(productivity_workbook.sheet_names)

Available sheets:
['Data', 'Indicator Metadata', 'Dataset Metadata']


### Inspecting the productivity data sheet

We preview the Data sheet to identify its headers, variables, and structure before importing it.

In [118]:
# Preview the Data sheet without assuming the header position.
productivity_preview = pd.read_excel(
    productivity_workbook,
    sheet_name="Data",
    header=None
)

print("Raw sheet dimensions:", productivity_preview.shape)
display(productivity_preview.iloc[:15, :12])

Raw sheet dimensions: (5743, 77)


,0,1,2,3,4,5,6,7,8,9,10,11
0,Economy ISO3,Economy Name,Indicator ID,Indicator,Attribute 1,Attribute 2,Attribute 3,Partner,1950-01-01,1951-01-01,1952-01-01,1953-01-01
1,AFG,Afghanistan,WB.ASPD.dlpe,Labor productivity growth rate (percent),Total,-,-,-,NaN,NaN,NaN,NaN
2,AFG,Afghanistan,WB.ASPD.lpxr,"Labor productivity level (GDP per employment, ...",Total,-,-,-,NaN,NaN,NaN,NaN
3,AGO,Angola,WB.ASPD.Employment,Employment (thousands),1.Agriculture,-,-,-,NaN,NaN,NaN,NaN
4,AGO,Angola,WB.ASPD.Employment,Employment (thousands),2.Mining,-,-,-,NaN,NaN,NaN,NaN
5,AGO,Angola,WB.ASPD.Employment,Employment (thousands),3.Manufacturing,-,-,-,NaN,NaN,NaN,NaN
6,AGO,Angola,WB.ASPD.Employment,Employment (thousands),4.Utilities,-,-,-,NaN,NaN,NaN,NaN
7,AGO,Angola,WB.ASPD.Employment,Employment (thousands),5.Construction,-,-,-,NaN,NaN,NaN,NaN
8,AGO,Angola,WB.ASPD.Employment,Employment (thousands),6.Trade services,-,-,-,NaN,NaN,NaN,NaN
9,AGO,Angola,WB.ASPD.Employment,Employment (thousands),7.Transport services,-,-,-,NaN,NaN,NaN,NaN


### Productivity variables and structure

The first row contains the column headers. We import the sheet correctly and inspect its indicators and time coverage.

In [120]:
# Import the Data sheet using its first row as the header.
productivity = pd.read_excel(
    productivity_workbook,
    sheet_name="Data",
    header=0
)

print("Dataset dimensions:", productivity.shape)
print("\nFirst columns:", productivity.columns[:8].tolist())

# Year headings use the format YYYY-01-01.
productivity_year_columns = [
    column for column in productivity.columns
    if str(column)[:4].isdigit() and str(column)[4:] == "-01-01"
]

print(
    "Period:",
    productivity_year_columns[0][:4],
    "-",
    productivity_year_columns[-1][:4]
)
print("Number of year columns:", len(productivity_year_columns))
print("Number of indicators:", productivity["Indicator ID"].nunique())

display(
    productivity[["Indicator ID", "Indicator"]]
    .drop_duplicates()
    .sort_values("Indicator ID")
    .reset_index(drop=True)
)

Dataset dimensions: (5742, 77)

Period:
1950 - 2018

Number of year columns:
69

Number of indicators:
9

Indicators:


,Indicator ID,Indicator
0,WB.ASPD.Employment,Employment (thousands)
1,WB.ASPD.Labor_productivity_PPP,Labor productivity (2011 international PPP exc...
2,WB.ASPD.Labor_productivity_real,"Labor productivity (2010 constant prices, loca..."
3,WB.ASPD.Value_added_nominal,"Nominal value added (current prices, local cur..."
4,WB.ASPD.Value_added_real,"Real value added (2010 constant prices, local ..."
5,WB.ASPD.capdeep,"Capital deepening, percent contribution"
6,WB.ASPD.dlpe,Labor productivity growth rate (percent)
7,WB.ASPD.dtfp,Total factor productivity (TFP) in log differe...
8,WB.ASPD.lpxr,"Labor productivity level (GDP per employment, ..."


### Selection of the productivity indicator

We use the annual labor productivity growth rate as our productivity variable. Before reshaping the data, we inspect whether each country has a unique observation.

In [121]:
# Select labor productivity growth
productivity_growth = productivity[
    productivity["Indicator ID"] == "WB.ASPD.dlpe"
].copy()

# Retain the European economies used in the other datasets
productivity_growth_europe = productivity_growth[
    productivity_growth["Economy ISO3"].isin(europe_codes)
].copy()

print("Dataset dimensions:", productivity_growth_europe.shape)
print(
    "Number of European economies:",
    productivity_growth_europe["Economy ISO3"].nunique()
)

# Inspect identifying columns and possible duplicate country rows
display(
    productivity_growth_europe[
        [
            "Economy ISO3",
            "Economy Name",
            "Attribute 1",
            "Attribute 2",
            "Attribute 3",
            "Partner"
        ]
    ]
    .sort_values("Economy Name")
    .reset_index(drop=True)
)

Dataset dimensions: (41, 77)
Number of European economies: 41


,Economy ISO3,Economy Name,Attribute 1,Attribute 2,Attribute 3,Partner
0,ALB,Albania,Total,-,-,-
1,AUT,Austria,Total,-,-,-
2,BLR,Belarus,Total,-,-,-
3,BEL,Belgium,Total,-,-,-
4,BIH,Bosnia and Herzegovina,Total,-,-,-
5,BGR,Bulgaria,Total,-,-,-
6,HRV,Croatia,Total,-,-,-
7,CYP,Cyprus,Total,-,-,-
8,CZE,Czechia,Total,-,-,-
9,DNK,Denmark,Total,-,-,-


### Reshaping productivity data

The productivity dataset contains one row per economy. We convert it from wide format to country-year panel format.

In [122]:
# Convert productivity data from wide to long format
productivity_europe_long = productivity_growth_europe.melt(
    id_vars=["Economy ISO3", "Economy Name"],
    value_vars=productivity_year_columns,
    var_name="Year",
    value_name="Productivity Growth"
)

# Extract the year from headings such as "1990-01-01"
productivity_europe_long["Year"] = (
    productivity_europe_long["Year"]
    .astype(str)
    .str[:4]
    .astype(int)
)

# Rename columns to match the other datasets
productivity_europe_long = productivity_europe_long.rename(
    columns={
        "Economy ISO3": "Country Code",
        "Economy Name": "Country Name"
    }
)

# Sort the country-year observations
productivity_europe_long = (
    productivity_europe_long
    .sort_values(["Country Name", "Year"])
    .reset_index(drop=True)
)

print(
    "Number of countries:",
    productivity_europe_long["Country Code"].nunique()
)
print(
    "Period:",
    productivity_europe_long["Year"].min(),
    "-",
    productivity_europe_long["Year"].max()
)
print("Dataset dimensions:", productivity_europe_long.shape)

display(productivity_europe_long.head(10))

Number of countries: 41
Period: 1950 - 2018
Dataset dimensions: (2829, 4)


,Country Code,Country Name,Year,Productivity Growth
0,ALB,Albania,1950,NaN
1,ALB,Albania,1951,NaN
2,ALB,Albania,1952,NaN
3,ALB,Albania,1953,NaN
4,ALB,Albania,1954,NaN
5,ALB,Albania,1955,NaN
6,ALB,Albania,1956,NaN
7,ALB,Albania,1957,NaN
8,ALB,Albania,1958,NaN
9,ALB,Albania,1959,NaN


## 6. Combined European panel

The four long-format datasets are merged on **country code and year**. The collected file covers the common calendar window **1960–2018** so that the cleaning notebook can show why the better-covered **2000–2018** period is selected for analysis. GDP defines the broad 46-economy staging panel; missing values are retained at this stage.

In [123]:
# Define the common calendar period
start_year = 1960
end_year = 2018

# Keep the common period in each dataset
gdp_common = gdp_europe_long[
    gdp_europe_long["Year"].between(start_year, end_year)
].copy()

investment_common = investment_europe_long[
    investment_europe_long["Year"].between(start_year, end_year)
].copy()

education_common = education_europe_long[
    education_europe_long["Year"].between(start_year, end_year)
].copy()

productivity_common = productivity_europe_long[
    productivity_europe_long["Year"].between(start_year, end_year)
].copy()

# Start from GDP to preserve all 46 European economies
europe_panel = gdp_common.copy()

# Add investment
europe_panel = europe_panel.merge(
    investment_common[
        ["Country Code", "Year", "Investment"]
    ],
    on=["Country Code", "Year"],
    how="left",
    validate="one_to_one"
)

# Add education expenditure
europe_panel = europe_panel.merge(
    education_common[
        ["Country Code", "Year", "Education Expenditure"]
    ],
    on=["Country Code", "Year"],
    how="left",
    validate="one_to_one"
)

# Add productivity growth
europe_panel = europe_panel.merge(
    productivity_common[
        ["Country Code", "Year", "Productivity Growth"]
    ],
    on=["Country Code", "Year"],
    how="left",
    validate="one_to_one"
)

# Sort the combined panel
europe_panel = (
    europe_panel
    .sort_values(["Country Name", "Year"])
    .reset_index(drop=True)
)

print("Number of countries:", europe_panel["Country Code"].nunique())
print("Period:", europe_panel["Year"].min(), "-", europe_panel["Year"].max())
print("Dataset dimensions:", europe_panel.shape)

display(europe_panel.head(10))

Number of countries: 46
Period: 1960 - 2018
Dataset dimensions: (2714, 7)


,Country Name,Country Code,Year,GDP Growth,Investment,Education Expenditure,Productivity Growth
0,Albania,ALB,1960,NaN,NaN,NaN,NaN
1,Albania,ALB,1961,NaN,NaN,NaN,NaN
2,Albania,ALB,1962,NaN,NaN,NaN,NaN
3,Albania,ALB,1963,NaN,NaN,NaN,NaN
4,Albania,ALB,1964,NaN,NaN,NaN,NaN
5,Albania,ALB,1965,NaN,NaN,NaN,NaN
6,Albania,ALB,1966,NaN,NaN,NaN,NaN
7,Albania,ALB,1967,NaN,NaN,NaN,NaN
8,Albania,ALB,1968,NaN,NaN,NaN,NaN
9,Albania,ALB,1969,NaN,NaN,NaN,NaN


## 7. Validation and export

The final checks confirm unique country–year keys, numeric variable types, and the amount of available data. Missing values are expected here and are handled explicitly in the cleaning notebook.

In [124]:
# Verify the country-year structure
duplicate_rows = europe_panel.duplicated(
    subset=["Country Code", "Year"]
).sum()

print("Duplicate country-year rows:", duplicate_rows)
print("\nData types:")
print(europe_panel.dtypes)

print("\nNon-missing observations:")
print(
    europe_panel[
        [
            "GDP Growth",
            "Investment",
            "Education Expenditure",
            "Productivity Growth"
        ]
    ].notna().sum()
)

print("\nMissing observations:")
print(
    europe_panel[
        [
            "GDP Growth",
            "Investment",
            "Education Expenditure",
            "Productivity Growth"
        ]
    ].isna().sum()
)

# Preview observations containing actual data
display(
    europe_panel.dropna(
        subset=[
            "GDP Growth",
            "Investment",
            "Education Expenditure",
            "Productivity Growth"
        ],
        how="all"
    ).head(10)
)

Duplicate country-year rows: 0

Data types:
Country Name                 str
Country Code                 str
Year                       int64
GDP Growth               float64
Investment               float64
Education Expenditure    float64
Productivity Growth      float64
dtype: object

Non-missing observations:
GDP Growth               1961
Investment               1636
Education Expenditure    1198
Productivity Growth      1456
dtype: int64

Missing observations:
GDP Growth                753
Investment               1078
Education Expenditure    1516
Productivity Growth      1258
dtype: int64


,Country Name,Country Code,Year,GDP Growth,Investment,Education Expenditure,Productivity Growth
20,Albania,ALB,1980,NaN,35.805835,NaN,0.21
21,Albania,ALB,1981,5.745635,36.386116,NaN,2.19
22,Albania,ALB,1982,2.948597,39.282900,NaN,-1.71
23,Albania,ALB,1983,1.104938,37.373346,NaN,-1.80
24,Albania,ALB,1984,-1.251597,32.925535,NaN,-3.34
25,Albania,ALB,1985,1.780644,33.931959,NaN,0.29
26,Albania,ALB,1986,5.637243,32.034835,NaN,2.25
27,Albania,ALB,1987,-0.787843,29.453870,NaN,-3.66
28,Albania,ALB,1988,-1.420040,29.957542,NaN,-3.10
29,Albania,ALB,1989,9.836549,32.877072,NaN,7.84


In [ ]:
output_file = processed_dir / "europe_panel_collected.csv"
europe_panel.to_csv(output_file, index=False)

print("Panel saved to:", output_file)

## Collection summary

The collected panel contains **2,714 country–year rows**, **46 European economies**, and four economic variables over 1960–2018. No imputation or observation deletion has been performed. The next notebook evaluates missingness, selects the final period, and creates the balanced core sample.